# Heart Disease Classification Project

This notebook performs the complete workflow: loading the dataset, cleaning, EDA, preprocessing, model training, evaluation, comparison, and prediction.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

## 2. Load the Dataset

In [ ]:
# If the CSV is in the same folder as this notebook, this is enough:
df = pd.read_csv('dataset_heart.csv')

print('Dataset loaded successfully!')
print('Shape:', df.shape)
df.head()

## 3. Inspect the Dataset

In [ ]:
print('Columns:')
print(df.columns.tolist())

print('\nData types:')
print(df.dtypes)

print('\nDataset information:')
df.info()

print('\nStatistical summary:')
df.describe()

## 4. Clean Column Names

In [ ]:
# Remove accidental spaces from column names
df.columns = df.columns.str.strip()
print(df.columns.tolist())

## 5. Check Missing Values and Duplicates

In [ ]:
print('Missing values:')
print(df.isnull().sum())

print('\nNumber of duplicate rows:', df.duplicated().sum())

In [ ]:
# Remove duplicate rows if any exist
df = df.drop_duplicates().reset_index(drop=True)
print('Shape after removing duplicates:', df.shape)

## 6. Explore the Target Variable

In [ ]:
print(df['heart disease'].value_counts().sort_index())

sns.countplot(data=df, x='heart disease')
plt.title('Heart Disease Class Distribution')
plt.xlabel('Heart Disease')
plt.ylabel('Count')
plt.show()

## 7. Exploratory Data Analysis (EDA)

The following plots help us understand the relationships between important features and the target.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='age', hue='heart disease', kde=True, bins=15)
plt.title('Age Distribution by Heart Disease Class')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='heart disease', y='max heart rate')
plt.title('Maximum Heart Rate by Heart Disease Class')
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

## 8. Prepare Features and Target

In [ ]:
X = df.drop('heart disease', axis=1)
y = df['heart disease'].copy()

# Convert the original labels (1 and 2) to binary labels (0 and 1)
y = y.map({1: 0, 2: 1})

print('Features shape:', X.shape)
print('Target shape:', y.shape)
print('Target values:', y.unique())

## 9. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)

## 10. Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Scaling completed successfully!')

## 11. Train Multiple Classification Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC()
}

results = {}

for name, model in models.items():
    if name in ['Logistic Regression', 'KNN', 'SVM']:
        model.fit(X_train_scaled, y_train)
        predictions = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    results[name] = accuracy
    print(f'{name}: {accuracy:.4f}')

## 12. Compare Model Accuracies

In [ ]:
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': list(results.values())
}).sort_values('Accuracy', ascending=False)

results_df

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=results_df, x='Accuracy', y='Model')
plt.xlim(0, 1)
plt.title('Model Accuracy Comparison')
plt.show()

## 13. Detailed Evaluation of the Best Model

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]

if best_model_name in ['Logistic Regression', 'KNN', 'SVM']:
    best_predictions = best_model.predict(X_test_scaled)
else:
    best_predictions = best_model.predict(X_test)

print('Best model based on this test split:', best_model_name)
print('\nClassification Report:')
print(classification_report(y_test, best_predictions))

In [ ]:
cm = confusion_matrix(y_test, best_predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 14. Example Prediction

The example below predicts the class of one existing test sample. The label is 0 for the original class 1 and 1 for the original class 2.

In [ ]:
sample = X_test.iloc[[0]]

if best_model_name in ['Logistic Regression', 'KNN', 'SVM']:
    sample_prediction = best_model.predict(scaler.transform(sample))[0]
else:
    sample_prediction = best_model.predict(sample)[0]

print('Predicted class:', sample_prediction)
print('Actual class   :', y_test.iloc[0])

## 15. Conclusion

- The dataset was loaded and inspected.
- Column names were cleaned and duplicate rows were checked.
- Missing values were checked.
- Exploratory data analysis was performed using plots and a correlation heatmap.
- The data was split into training and testing sets.
- Several classification models were trained and compared.
- Accuracy, classification report, and confusion matrix were used for evaluation.

**Note:** Model accuracy depends on the train/test split and should not be treated as a clinical conclusion.